# 📖 Notebook 3: Retention Policies & Downsampling

Storing 30-second resolution data forever is expensive and pointless — nobody needs millisecond precision for last year's CPU data. This notebook shows how to:

1. **Automatically delete old data** with retention policies
2. **Pre-compute rollups** with continuous aggregates (downsampling)
3. Layer both together for a production-grade data lifecycle

## Learning Objectives

By the end of this notebook you will understand:
- How TimescaleDB retention policies drop old chunks automatically
- How continuous aggregates pre-compute time-bucketed summaries in the background
- A real-world tiered storage strategy: raw → 5-min → 1-hour rollups
- The storage savings from downsampling

## 🛠️ Setup

```bash
cd 03-technologies/databases/time-series-databases
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='.*pandas only supports SQLAlchemy.*')

import psycopg2
import pandas as pd
import matplotlib.pyplot as plt

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "tsdb_demo",
    "user": "demo",
    "password": "demo"
}

def run_query(sql, params=None):
    with psycopg2.connect(**DB_CONFIG) as conn:
        return pd.read_sql_query(sql, conn, params=params)

def run_exec(sql):
    # NOTE: autocommit must be set BEFORE any statement runs. Using the
    # connection as a context manager (`with psycopg2.connect(...) as conn`)
    # opens a transaction, and flipping autocommit after that does not close
    # it -- so statements TimescaleDB refuses to run inside a transaction
    # (CALL refresh_continuous_aggregate, CALL run_job) fail with
    # "ActiveSqlTransaction: ... cannot run inside a transaction block".
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
    finally:
        conn.close()

print("✅ Connected!")

---
## 1 — Current Storage Baseline

Before we change anything, let's see how much space our raw data uses.

In [ ]:
size = run_query("""
    SELECT
        hypertable_name,
        pg_size_pretty(hypertable_size('metrics')) AS total_size,
        num_chunks
    FROM timescaledb_information.hypertables
    WHERE hypertable_name = 'metrics'
""")
size

In [ ]:
# Size per chunk (partition)
chunks = run_query("""
    -- Chunk sizes do NOT live on timescaledb_information.chunks (that view has
    -- no total_bytes column in TimescaleDB 2.x -- asking for it raises
    -- "column total_bytes does not exist"). Sizes come from the
    -- chunks_detailed_size() function, joined back on chunk_name.
    SELECT
        c.chunk_name,
        c.range_start,
        c.range_end,
        pg_size_pretty(d.total_bytes) AS size
    FROM timescaledb_information.chunks c
    JOIN chunks_detailed_size('metrics') d USING (chunk_schema, chunk_name)
    WHERE c.hypertable_name = 'metrics'
    ORDER BY c.range_start
""")
print(f"Chunks: {len(chunks)}")
chunks

---
## 2 — Retention Policies: Auto-Delete Old Data

A **retention policy** tells TimescaleDB: *"Automatically drop chunks older than X."*

Because data is already partitioned by time, dropping a chunk is like deleting a file — instant, no expensive row-by-row `DELETE`.  
This is the *"retention becomes trivial"* insight from the article.

Let's add a 5-day retention policy on raw data.

> **Note**: In production you'd keep raw data for 7-30 days. We use 5 days here so we can see the effect immediately with our 7-day seed data.

In [ ]:
# Count rows BEFORE retention
before = run_query("SELECT count(*) AS n FROM metrics")
print(f"Rows before retention: {before['n'].iloc[0]:,}")

# Add a retention policy: drop chunks older than 5 days
run_exec("""
    SELECT add_retention_policy('metrics', drop_after => INTERVAL '5 days', if_not_exists => true)
""")
print("✅ Retention policy added (drop chunks older than 5 days)")

In [ ]:
# The policy runs on a schedule. Let's trigger it manually to see the effect now.
def run_job_now(proc_name):
    """Trigger a TimescaleDB background job immediately, by name.

    You cannot write `CALL run_job((SELECT job_id ...))` -- Postgres rejects a
    subquery in a CALL argument with "cannot use subquery in CALL argument".
    So look the job id up first, then pass it as a literal.
    """
    jobs = run_query(
        f"SELECT job_id FROM timescaledb_information.jobs WHERE proc_name = '{proc_name}'"
    )
    if jobs.empty:
        raise RuntimeError(f"No {proc_name} job is registered -- was the policy added?")
    job_id = int(jobs["job_id"].iloc[0])
    run_exec(f"CALL run_job({job_id})")
    return job_id


run_job_now("policy_retention")

after = run_query("SELECT count(*) AS n FROM metrics")
print(f"Rows after retention:  {after['n'].iloc[0]:,}")
print(f"Rows dropped:          {before['n'].iloc[0] - after['n'].iloc[0]:,}")

In [ ]:
# Check which chunks survived
remaining = run_query("""
    SELECT c.chunk_name, c.range_start, c.range_end, pg_size_pretty(d.total_bytes) AS size
    FROM timescaledb_information.chunks c
    JOIN chunks_detailed_size('metrics') d USING (chunk_schema, chunk_name)
    WHERE c.hypertable_name = 'metrics'
    ORDER BY c.range_start
""")
print(f"Chunks remaining: {len(remaining)} (older chunks were dropped)")
remaining

Dropping a chunk is **O(1)** — it just removes the underlying file. Compare this to a `DELETE FROM metrics WHERE time < X` on a regular table, which would need to scan and remove rows one by one.

---
## 3 — Continuous Aggregates: Pre-Computed Rollups

A **continuous aggregate** is a materialized view that TimescaleDB keeps up-to-date automatically.  
It pre-computes `time_bucket` + aggregation so that dashboards read from the rollup instead of scanning raw data.

This is the **downsampling** concept from the article:
- Raw data (30s) → **5-minute rollup** → **1-hour rollup**
- Each level stores min, max, avg, count so you can answer most queries without touching raw data.

### Create a 5-minute rollup

In [ ]:
run_exec("""
    CREATE MATERIALIZED VIEW IF NOT EXISTS metrics_5min
    WITH (timescaledb.continuous) AS
    SELECT
        time_bucket('5 minutes', time) AS bucket,
        host,
        region,
        metric_name,
        avg(value)   AS avg_value,
        min(value)   AS min_value,
        max(value)   AS max_value,
        count(*)     AS sample_count
    FROM metrics
    GROUP BY bucket, host, region, metric_name
    WITH NO DATA
""")
print("✅ Created continuous aggregate: metrics_5min")

In [ ]:
# Manually refresh to backfill historical data
run_exec("""
    CALL refresh_continuous_aggregate('metrics_5min', now() - interval '7 days', now())
""")
print("✅ Backfilled metrics_5min")

run_query("SELECT count(*) AS rows FROM metrics_5min")

In [ ]:
# Set up an auto-refresh policy so new data is rolled up automatically
run_exec("""
    SELECT add_continuous_aggregate_policy('metrics_5min',
        start_offset  => INTERVAL '1 hour',
        end_offset    => INTERVAL '5 minutes',
        schedule_interval => INTERVAL '5 minutes',
        if_not_exists => true
    )
""")
print("✅ Auto-refresh policy added — metrics_5min will stay up to date")

### Create a 1-hour rollup

We can even build a rollup **on top of another rollup** for very long-term queries.

In [ ]:
run_exec("""
    CREATE MATERIALIZED VIEW IF NOT EXISTS metrics_1hour
    WITH (timescaledb.continuous) AS
    SELECT
        time_bucket('1 hour', bucket) AS bucket,
        host,
        region,
        metric_name,
        avg(avg_value)   AS avg_value,
        min(min_value)   AS min_value,
        max(max_value)   AS max_value,
        sum(sample_count) AS sample_count
    FROM metrics_5min
    -- Group by the time_bucket EXPRESSION, not by the output alias. `bucket` is
    -- also a real column of metrics_5min, so `GROUP BY bucket` binds to that
    -- source column instead of the alias -- TimescaleDB then sees no bucketing
    -- and rejects the view with
    --   "continuous aggregate view must include a valid time bucket function".
    GROUP BY time_bucket('1 hour', bucket), host, region, metric_name
    WITH NO DATA
""")
print("✅ Created continuous aggregate: metrics_1hour")

run_exec("""
    CALL refresh_continuous_aggregate('metrics_1hour', now() - interval '7 days', now())
""")
print("✅ Backfilled metrics_1hour")

run_exec("""
    SELECT add_continuous_aggregate_policy('metrics_1hour',
        start_offset  => INTERVAL '3 hours',
        end_offset    => INTERVAL '1 hour',
        schedule_interval => INTERVAL '1 hour',
        if_not_exists => true
    )
""")
print("✅ Auto-refresh policy added for metrics_1hour")

---
## 4 — Querying Rollups vs. Raw Data

Now we have three tiers:

| Tier | Resolution | Best For |
|------|-----------|----------|
| `metrics` | 30 seconds | Debugging recent issues |
| `metrics_5min` | 5 minutes | Hourly/daily dashboards |
| `metrics_1hour` | 1 hour | Weekly/monthly trends |

Let's compare query speed across all three.

In [ ]:
import time as _time

queries = {
    "raw (metrics)": """
        SELECT date_trunc('hour', time) AS hour, avg(value)
        FROM metrics
        WHERE host = 'server-1' AND metric_name = 'cpu_usage'
          AND time > now() - interval '3 days'
        GROUP BY hour ORDER BY hour
    """,
    "5-min rollup": """
        SELECT date_trunc('hour', bucket) AS hour, avg(avg_value)
        FROM metrics_5min
        WHERE host = 'server-1' AND metric_name = 'cpu_usage'
          AND bucket > now() - interval '3 days'
        GROUP BY hour ORDER BY hour
    """,
    "1-hour rollup": """
        SELECT bucket AS hour, avg_value
        FROM metrics_1hour
        WHERE host = 'server-1' AND metric_name = 'cpu_usage'
          AND bucket > now() - interval '3 days'
        ORDER BY hour
    """
}

for label, sql in queries.items():
    start = _time.perf_counter()
    df = run_query(sql)
    elapsed = (_time.perf_counter() - start) * 1000
    print(f"{label:20s} → {elapsed:6.1f} ms  ({len(df)} rows)")

The rollup queries are faster because they scan **far fewer rows** — the aggregation is already done.

---
## 4b — Native Compression: Shrink Old Data 10–20×

Downsampling discards detail. **Compression keeps every point** but packs it into columnar blocks with *delta-of-delta* encoding for timestamps and dictionary/Gorilla encoding for values — exactly the techniques the article lists under "building blocks". On monitoring data, 10–20× size reduction is typical.

TimescaleDB compression is **chunk-level**: you pick an age threshold, and chunks older than that get rewritten in compressed form.

Trade-offs to teach beginners:

| ✅ Compressed chunks | ❌ Compressed chunks |
|----------------------|----------------------|
| Much smaller on disk | No `UPDATE` / `DELETE` without decompressing first |
| Faster bulk scans (fewer pages read) | Single-row lookups slightly slower |
| Still queryable via normal SQL | Ingest into a compressed chunk needs decompress-then-recompress |

> **Rule of thumb:** compress data that is *cold* (not being written anymore). For 30-second metrics, "older than a few hours" is usually safe.


In [ ]:
# Enable compression on the hypertable and pick which columns to segment by.
# `segmentby` columns are stored uncompressed to keep WHERE filters fast.
# `orderby` defines the row order inside each compressed block — ordering by time
# makes timestamp delta-encoding maximally effective.
run_exec("""
    ALTER TABLE metrics SET (
        timescaledb.compress,
        timescaledb.compress_segmentby = 'host, metric_name',
        timescaledb.compress_orderby   = 'time DESC'
    )
""")
print("✅ Compression enabled on 'metrics'")


In [ ]:
# Tell TimescaleDB to automatically compress chunks older than 2 days.
# The policy runs on a schedule; we also trigger it manually to see the effect now.
run_exec("""
    SELECT add_compression_policy('metrics', INTERVAL '2 days', if_not_exists => true)
""")
run_job_now("policy_compression")
print("✅ Compression policy added and run")


In [ ]:
# Inspect per-chunk compression status and the before/after size savings.
comp = run_query("""
    SELECT
        chunk_schema || '.' || chunk_name         AS chunk,
        range_start::date                          AS day,
        is_compressed,
        pg_size_pretty(before_compression_total_bytes) AS before_size,
        pg_size_pretty(after_compression_total_bytes)  AS after_size
    FROM chunk_compression_stats('metrics')
    JOIN timescaledb_information.chunks USING (chunk_schema, chunk_name)
    ORDER BY range_start
""")
comp


In [ ]:
# Hypertable-level compression summary
summary = run_query("""
    SELECT
        pg_size_pretty(before_compression_total_bytes) AS before_total,
        pg_size_pretty(after_compression_total_bytes)  AS after_total,
        round(
            before_compression_total_bytes::numeric
            / NULLIF(after_compression_total_bytes, 0),
            1
        ) AS compression_ratio
    FROM hypertable_compression_stats('metrics')
""")
summary


---
## 5 — Storage Comparison

Let's see how much space each tier uses.

In [ ]:
storage = run_query("""
    -- The view exposes the materialization hypertable's NAME, not its size
    -- (asking for materialization_hypertable_size raises "column ... does not
    -- exist"). Resolve the name to a regclass and measure it with
    -- hypertable_size().
    SELECT
        view_name AS name,
        pg_size_pretty(
            hypertable_size(
                format('%I.%I', materialization_hypertable_schema,
                                materialization_hypertable_name)::regclass
            )
        ) AS size
    FROM timescaledb_information.continuous_aggregates
""")

raw_size = run_query("SELECT pg_size_pretty(hypertable_size('metrics')) AS size")
print(f"{'metrics (raw)':25s} : {raw_size['size'].iloc[0]}")
for _, row in storage.iterrows():
    print(f"{row['name']:25s} : {row['size']}")

In [ ]:
# Row counts across tiers
for table in ['metrics', 'metrics_5min', 'metrics_1hour']:
    count_col = 'bucket' if table != 'metrics' else 'time'
    n = run_query(f"SELECT count(*) AS n FROM {table}")
    print(f"{table:20s} : {n['n'].iloc[0]:>10,} rows")

The 1-hour rollup is **tiny** compared to raw data but can still answer most long-range dashboard queries.

---
## 6 — The Full Data Lifecycle

Here is a production-grade tiered strategy:

```
┌─────────────────────────────────────────────────────────────┐
│              DATA LIFECYCLE                                  │
├──────────────┬──────────────┬───────────────────────────────┤
│ Age          │ Tier         │ Policy                        │
├──────────────┼──────────────┼───────────────────────────────┤
│ 0 – 2 days   │ Raw (30s)    │ Hot, uncompressed             │
│ 2 – 7 days   │ Raw (30s)    │ Compressed (10–20× smaller)   │
│ 0 – 30 days │ 5-min rollup │ Continuous aggregate          │
│ 0 – 1 year  │ 1-hour rollup│ Continuous aggregate          │
├──────────────┼──────────────┼───────────────────────────────┤
│ > 7 days     │ Raw          │ Retention policy → DROP       │
│ > 30 days    │ 5-min rollup │ Retention policy → DROP       │
│ > 1 year     │ 1-hour rollup│ Retention policy → DROP       │
└──────────────┴──────────────┴───────────────────────────────┘
```

The key insight: **you never lose the ability to answer queries** about historical data — you just answer them at lower resolution.  
"What was average CPU last Tuesday?" → answered by the 5-min rollup.  
"What was average CPU in January?" → answered by the 1-hour rollup.

---
## 7 — Visualize All Three Tiers

Let's plot the same time range from all three tiers to see the resolution difference.

In [ ]:
raw = run_query("""
    SELECT time AS ts, value AS val FROM metrics
    WHERE host = 'server-1' AND metric_name = 'cpu_usage'
      AND time > now() - interval '24 hours'
    ORDER BY ts
""")

five = run_query("""
    SELECT bucket AS ts, avg_value AS val FROM metrics_5min
    WHERE host = 'server-1' AND metric_name = 'cpu_usage'
      AND bucket > now() - interval '24 hours'
    ORDER BY ts
""")

hour = run_query("""
    SELECT bucket AS ts, avg_value AS val FROM metrics_1hour
    WHERE host = 'server-1' AND metric_name = 'cpu_usage'
      AND bucket > now() - interval '24 hours'
    ORDER BY ts
""")

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(raw['ts'], raw['val'], linewidth=0.4, alpha=0.8)
axes[0].set_title(f'Raw 30s data ({len(raw)} points)')
axes[0].set_ylabel('CPU %')

axes[1].plot(five['ts'], five['val'], linewidth=1, color='orange')
axes[1].set_title(f'5-min rollup ({len(five)} points)')
axes[1].set_ylabel('CPU %')

axes[2].step(hour['ts'], hour['val'], linewidth=1.5, color='green', where='mid')
axes[2].set_title(f'1-hour rollup ({len(hour)} points)')
axes[2].set_ylabel('CPU %')
axes[2].set_xlabel('Time')

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.suptitle('Same data at three resolutions — server-1 CPU (24h)', fontsize=14)
plt.tight_layout()
plt.show()

---
## 8 — Cleanup: View Active Policies

Let's see all the automated jobs TimescaleDB is now running for us.

In [ ]:
jobs = run_query("""
    SELECT
        job_id,
        proc_name,
        hypertable_name,
        schedule_interval,
        next_start
    FROM timescaledb_information.jobs
    WHERE hypertable_name IS NOT NULL
    ORDER BY job_id
""")
jobs

---
## 🧠 Key Takeaways

1. **Retention policies** auto-drop old chunks — instant and free compared to row-level `DELETE`.
2. **Continuous aggregates** are materialized views that TimescaleDB refreshes automatically — they pre-compute `time_bucket` + aggregation.
3. **Native compression** shrinks cold chunks 10–20× with delta-of-delta + Gorilla encoding — same data, far less disk.
4. **Tiered storage** (raw → 5-min → 1-hour) trades precision for efficiency on *even older* data.
5. Rollups preserve `min`, `max`, `avg`, `count` so you can still answer most analytical queries.
6. This pattern is universal in monitoring systems: Prometheus, Datadog, Grafana Cloud all do the same thing.

## 🎓 Congratulations!

You've completed the Time-Series Databases lab series. You now understand:
- How time-series data is modeled (Notebook 1)
- How to aggregate and analyze it with windowed queries (Notebook 2)
- How to manage its lifecycle with retention and downsampling (Notebook 3)

These are the same building blocks behind systems like Prometheus, InfluxDB, and Datadog.